In [13]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add scripts directory to path
sys.path.append(os.path.abspath("../../scripts"))

# Import modules
import data_pipeline
import xgb_scripts

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
unique_frequencies = [
    2.54e-01, 3.40e-01, 4.56e-01, 6.12e-01, 8.22e-01, 9.99e-01, 1.10e+00, 1.33e+00,
    1.48e+00, 1.78e+00, 1.99e+00, 2.37e+00, 2.66e+00, 3.16e+00, 3.57e+00, 4.22e+00,
    4.80e+00, 5.62e+00, 6.43e+00, 7.50e+00, 8.64e+00, 1.00e+01, 1.16e+01, 1.33e+01,
    1.55e+01, 1.78e+01, 2.09e+01, 2.37e+01, 2.80e+01, 3.16e+01, 3.75e+01, 4.22e+01,
    5.03e+01, 5.62e+01, 6.76e+01, 7.50e+01, 9.06e+01, 1.02e+02, 1.22e+02, 1.35e+02,
    1.63e+02, 1.78e+02, 2.19e+02, 2.37e+02, 2.94e+02, 3.16e+02, 3.94e+02, 4.22e+02,
    5.29e+02, 5.64e+02, 7.10e+02, 7.50e+02, 9.52e+02, 1.00e+03, 1.28e+03, 1.33e+03,
    1.71e+03, 1.78e+03, 2.30e+03, 2.37e+03, 3.09e+03, 3.16e+03, 4.14e+03, 4.22e+03,
    5.56e+03, 5.62e+03, 7.45e+03, 7.50e+03, 1.00e+04
]

## Exploring relevant frequencies

## Systematic Frequency Selection Methodology

Following research best practices to determine optimal frequencies for our specific dataset.
This approach combines multiple statistical and physical criteria rather than just copying literature values.

## Comprehensive Feature Importance Analysis Using All Cycles Model

Loading the XGBoost model trained on ALL CYCLES (1-267) with binning + all frequencies to understand which specific frequencies and components are most important across the complete battery degradation spectrum - from early life through end-of-life conditions.

In [15]:
# Load the high-performing model trained on ALL CYCLES (1-267)
import joblib

models = joblib.load("../../models/xgb_binning_all_freq_all_cycles.pkl")
print(f"\nSuccessfully loaded XGBoost ensemble with {len(models)} models")
print(f"Model type: {type(models[0])}")


Successfully loaded XGBoost ensemble with 10 models
Model type: <class 'xgboost.sklearn.XGBRegressor'>


In [16]:
# Load data with ALL CYCLES to match the saved model configuration
X_train, X_test, y_train, y_test = data_pipeline.load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    method="bin_and_split",
)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Train capacity range: {y_train.min():.1f} - {y_train.max():.1f} mAh")
print(f"Test capacity range: {y_test.min():.1f} - {y_test.max():.1f} mAh")
print(f"Data loaded: X_train shape = {X_train.shape}, X_test shape = {X_test.shape}")

# Get feature importance from the loaded models
feature_importance = np.mean([model.feature_importances_ for model in models], axis=0)
print(f"Feature importance extracted from {len(models)} models")

# Sort features by importance
feature_indices = np.argsort(feature_importance)[::-1]  # Descending order
print(f"\nTop 10 most important features:")
for i in range(10):
    idx = feature_indices[i]
    print(f"{i+1:2d}. Feature {idx:3d}: Importance = {feature_importance[idx]:.4f}")

# Show cumulative importance
cumulative_importance = np.cumsum(feature_importance[feature_indices])
total_importance = np.sum(feature_importance)

print(f"\nCumulative importance breakdown:")
for n_features in [5, 10, 15, 20, 30]:
    if n_features <= len(feature_importance):
        pct = cumulative_importance[n_features-1] / total_importance * 100
        print(f"Top {n_features:2d} features capture {pct:.1f}% of total importance")

X_train: (290, 140), y_train: (290,)
X_test: (152, 140), y_test: (152,)
Train capacity range: 80.7 - 4070.0 mAh
Test capacity range: 78.0 - 3850.0 mAh
X_train: (290, 140), y_train: (290,)
X_test: (152, 140), y_test: (152,)
Train capacity range: 80.7 - 4070.0 mAh
Test capacity range: 78.0 - 3850.0 mAh
Data loaded: X_train shape = (290, 140), X_test shape = (152, 140)
Feature importance extracted from 10 models

Top 10 most important features:
 1. Feature 138: Importance = 0.6242
 2. Feature  69: Importance = 0.1878
 3. Feature  90: Importance = 0.0949
 4. Feature 139: Importance = 0.0132
 5. Feature 100: Importance = 0.0126
 6. Feature  86: Importance = 0.0097
 7. Feature  89: Importance = 0.0062
 8. Feature  88: Importance = 0.0062
 9. Feature  68: Importance = 0.0048
10. Feature  16: Importance = 0.0046

Cumulative importance breakdown:
Top  5 features capture 93.3% of total importance
Top 10 features capture 96.4% of total importance
Top 15 features capture 98.3% of total importance


In [17]:
feature_details = []
for i in range(len(feature_importance)):
    if i < 69:  # Real impedance
        freq = unique_frequencies[i]
        feature_type = "Real"
    elif i < 138:  # Imaginary impedance  
        freq = unique_frequencies[i - 69]
        feature_type = "Imaginary"
    else:  # Action vector
        freq = None
        feature_type = "Action"
    
    feature_details.append({
        'idx': i,
        'type': feature_type,
        'frequency': freq,
        'importance': feature_importance[i]
    })

# Sort by importance (descending)
feature_details.sort(key=lambda x: x['importance'], reverse=True)

print("ALL FEATURES RANKED BY IMPORTANCE:")
print("Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total")

total_imp = sum(feature_importance)
cumulative = 0

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    if feature['frequency'] is not None:
        freq_str = f"{feature['frequency']:>8.2f}"
    else:
        freq_str = "   Action"
    
    print(f"{rank:4d} | {feature['idx']:7d} | {feature['type']:<9} | {freq_str} | {feature['importance']:10.4f} | {pct_individual:5.1f}% ({pct_cumulative:5.1f}%)")


ALL FEATURES RANKED BY IMPORTANCE:
Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total
   1 |     138 | Action    |    Action |     0.6242 |  62.4% ( 62.4%)
   2 |      69 | Imaginary |     0.25 |     0.1878 |  18.8% ( 81.2%)
   3 |      90 | Imaginary |    10.00 |     0.0949 |   9.5% ( 90.7%)
   4 |     139 | Action    |    Action |     0.0132 |   1.3% ( 92.0%)
   5 |     100 | Imaginary |    42.20 |     0.0126 |   1.3% ( 93.3%)
   6 |      86 | Imaginary |     5.62 |     0.0097 |   1.0% ( 94.2%)
   7 |      89 | Imaginary |     8.64 |     0.0062 |   0.6% ( 94.9%)
   8 |      88 | Imaginary |     7.50 |     0.0062 |   0.6% ( 95.5%)
   9 |      68 | Real      | 10000.00 |     0.0048 |   0.5% ( 96.0%)
  10 |      16 | Real      |     4.80 |     0.0046 |   0.5% ( 96.4%)
  11 |      19 | Real      |     7.50 |     0.0045 |   0.5% ( 96.9%)
  12 |     106 | Imaginary |   102.00 |     0.0042 |   0.4% ( 97.3%)
  13 |     127 | Imaginary |  2300.00 |     0.0039 |   0.4% ( 97.

In [18]:
# Save comprehensive feature analysis to files - ALL CYCLES VERSION
import pandas as pd
import json
import os

# Create results directory if it doesn't exist
results_dir = "../../results"
os.makedirs(results_dir, exist_ok=True)

# Prepare data for saving - convert numpy types to native Python types for JSON compatibility
feature_data = []
cumulative = 0

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    feature_data.append({
        'rank': rank,
        'feature_index': int(feature['idx']),
        'feature_type': feature['type'],
        'frequency_hz': float(feature['frequency']) if feature['frequency'] is not None else 'Action_Vector',
        'importance': float(feature['importance']),
        'importance_percent': float(pct_individual),
        'cumulative_percent': float(pct_cumulative),
        'is_literature_range_1_10_hz': (
            True if feature['frequency'] is not None and 1 <= feature['frequency'] <= 10 
            else False if feature['frequency'] is not None 
            else None
        ),
        'frequency_range': (
            'Low (1-10 Hz)' if feature['frequency'] is not None and 1 <= feature['frequency'] <= 10
            else 'Mid (10-1000 Hz)' if feature['frequency'] is not None and 10 < feature['frequency'] <= 1000
            else 'High (>1000 Hz)' if feature['frequency'] is not None and feature['frequency'] > 1000
            else 'Action Vector'
        )
    })

# Save as CSV
df_features = pd.DataFrame(feature_data)
csv_path = os.path.join(results_dir, "feature_relevance_all_cycles.csv")
df_features.to_csv(csv_path, index=False)

# Save as JSON with proper type conversion
json_path = os.path.join(results_dir, "feature_relevance_all_cycles.json")
with open(json_path, 'w') as f:
    json.dump({
        'metadata': {
            'model_type': 'XGBoost_ensemble_binning_all_frequencies_ALL_CYCLES',
            'cycle_range': '1-267 (all available cycles)',
            'total_features': int(len(feature_importance)),
            'model_performance': 'R² ~0.9255 (all cycles model)',
            'analysis_date': '2025-10-10',
            'data_samples': f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}',
            'capacity_range': f'{y_train.min():.0f}-{y_train.max():.0f} mAh',
            'top_5_capture_percent': f"{float(cumulative_importance[4]/total_importance*100):.1f}%",
            'top_10_capture_percent': f"{float(cumulative_importance[9]/total_importance*100):.1f}%"
        },
        'feature_analysis': feature_data
    }, f, indent=2)

print(f"ALL CYCLES Feature relevance analysis saved:")
print(f"CSV: {csv_path}")
print(f"JSON: {json_path}")

# Summary statistics
zero_importance = len([f for f in feature_data if f['importance'] == 0])
significant_features = len([f for f in feature_data if f['importance_percent'] >= 1.0])
literature_range_features = len([f for f in feature_data if f['is_literature_range_1_10_hz'] == True])

print(f"\nALL CYCLES Analysis Summary:")
print(f"Total features: {len(feature_data)}")
print(f"Zero importance: {zero_importance}")
print(f"Significant (>=1%): {significant_features}")
print(f"Literature range (1-10 Hz): {literature_range_features}")

# Top feature sets
top_5_features = feature_data[:5]
print(f"\nTop 5 features capture {top_5_features[-1]['cumulative_percent']:.1f}% importance")
print(f"Top 10 features capture {feature_data[9]['cumulative_percent']:.1f}% importance")

# Analyze frequency distribution in top features
print(f"\nFrequency range distribution in top 20 features:")
top_20 = feature_data[:20]
freq_ranges = {}
for feature in top_20:
    range_name = feature['frequency_range']
    if range_name not in freq_ranges:
        freq_ranges[range_name] = []
    freq_ranges[range_name].append(feature)

for range_name, features in freq_ranges.items():
    print(f"{range_name}: {len(features)} features ({len(features)/20*100:.1f}%)")

ALL CYCLES Feature relevance analysis saved:
CSV: ../../results/feature_relevance_all_cycles.csv
JSON: ../../results/feature_relevance_all_cycles.json

ALL CYCLES Analysis Summary:
Total features: 140
Zero importance: 0
Significant (>=1%): 5
Literature range (1-10 Hz): 32

Top 5 features capture 93.3% importance
Top 10 features capture 96.4% importance

Frequency range distribution in top 20 features:
Action Vector: 6 features (30.0%)
Low (1-10 Hz): 8 features (40.0%)
Mid (10-1000 Hz): 4 features (20.0%)
High (>1000 Hz): 2 features (10.0%)


## Comparison: All Cycles vs Limited Cycles Feature Importance

Analyzing how feature importance patterns change when including the full degradation spectrum (cycles 1-267) versus limited cycles (1-200).

In [19]:
# Compare with previous limited cycles analysis (if available)
previous_results_path = "../../results/feature_relevance.csv"

if os.path.exists(previous_results_path):
    print("=== COMPARISON: ALL CYCLES vs LIMITED CYCLES FEATURE IMPORTANCE ===\n")
    
    # Load previous analysis (limited cycles)
    df_limited = pd.read_csv(previous_results_path)
    df_all_cycles = pd.DataFrame(feature_data)
    
    print("TOP 10 FEATURES COMPARISON:")
    print("Rank | All Cycles (1-267)           | Limited Cycles (1-200)      | Change")
    print("     | Feature | Type | Freq | %   | Feature | Type | Freq | %   |")
    print("-----|---------|------|------|-----|---------|------|------|-----|-------")
    
    for i in range(10):
        # All cycles data
        all_feat = df_all_cycles.iloc[i]
        all_freq = f"{all_feat['frequency_hz']:.1f}" if isinstance(all_feat['frequency_hz'], (int, float)) else "Action"
        
        # Limited cycles data
        if i < len(df_limited):
            lim_feat = df_limited.iloc[i]
            lim_freq = f"{lim_feat['frequency_hz']:.1f}" if isinstance(lim_feat['frequency_hz'], (int, float)) else "Action"
            
            # Check if same feature
            same_feature = (all_feat['feature_index'] == lim_feat['feature_index'])
            change_indicator = "✓" if same_feature else "✗"
            
            # Format limited cycles feature index properly
            lim_feat_idx = f"{lim_feat['feature_index']:7d}"
            lim_feat_type = f"{lim_feat['feature_type'][:4]:4s}"
            lim_importance = f"{lim_feat['importance_percent']:3.1f}"
        else:
            lim_feat_idx = "    N/A"
            lim_feat_type = " N/A"  
            lim_freq = " N/A"
            lim_importance = "0.0"
            change_indicator = "✗"
        
        print(f"{i+1:4d} | {all_feat['feature_index']:7d} | {all_feat['feature_type'][:4]:4s} | {all_freq:>4s} | {all_feat['importance_percent']:3.1f} | "
              f"{lim_feat_idx} | {lim_feat_type} | {lim_freq:>4s} | {lim_importance} | {change_indicator}")
    
    # Analyze frequency range shifts
    print(f"\n=== FREQUENCY RANGE ANALYSIS ===")
    
    # All cycles frequency ranges in top 20
    all_top_20 = df_all_cycles.head(20)
    all_freq_dist = all_top_20['frequency_range'].value_counts()
    
    # Limited cycles frequency ranges in top 20  
    lim_top_20 = df_limited.head(20)
    lim_freq_dist = lim_top_20['frequency_range'].value_counts()
    
    print("Frequency Range Distribution in Top 20 Features:")
    print("Range             | All Cycles | Limited Cycles | Change")
    print("------------------|------------|----------------|--------")
    
    all_ranges = set(all_freq_dist.index) | set(lim_freq_dist.index)
    for range_name in sorted(all_ranges):
        all_count = all_freq_dist.get(range_name, 0)
        lim_count = lim_freq_dist.get(range_name, 0)
        change = all_count - lim_count
        change_str = f"+{change}" if change > 0 else str(change) if change < 0 else "0"
        print(f"{range_name:<17} | {all_count:10d} | {lim_count:14d} | {change_str:>6s}")
    
    # Calculate feature stability
    matching_features = 0
    for i in range(min(10, len(df_limited))):
        if df_all_cycles.iloc[i]['feature_index'] == df_limited.iloc[i]['feature_index']:
            matching_features += 1
    
    # Key insights
    print(f"\n=== KEY INSIGHTS ===")
    print(f"1. Feature Stability: {matching_features}/10 top features remain the same")
    print("2. Frequency Shifts: Which frequency ranges become more/less important")
    print("3. Late-Stage Impact: How including cycles 201-267 changes priorities")
    
    if matching_features >= 7:
        print("   → HIGH stability: Core features are consistent across cycle ranges")
    elif matching_features >= 4:
        print("   → MODERATE stability: Some features change with extended cycles")
    else:
        print("   → LOW stability: Late-stage cycles significantly alter feature importance")
    
else:
    print("Previous feature analysis (limited cycles) not found.")
    print("Run the limited cycles analysis first to enable comparison.")

=== COMPARISON: ALL CYCLES vs LIMITED CYCLES FEATURE IMPORTANCE ===

TOP 10 FEATURES COMPARISON:
Rank | All Cycles (1-267)           | Limited Cycles (1-200)      | Change
     | Feature | Type | Freq | %   | Feature | Type | Freq | %   |
-----|---------|------|------|-----|---------|------|------|-----|-------
   1 |     138 | Acti | Action | 62.4 |       4 | Real | Action | 43.8 | ✗
   2 |      69 | Imag |  0.3 | 18.8 |      71 | Imag | Action | 22.0 | ✗
   3 |      90 | Imag | 10.0 | 9.5 |     138 | Acti | Action | 14.6 | ✗
   4 |     139 | Acti | Action | 1.3 |      23 | Real | Action | 8.1 | ✗
   5 |     100 | Imag | 42.2 | 1.3 |      68 | Real | Action | 5.2 | ✗
   6 |      86 | Imag |  5.6 | 1.0 |     139 | Acti | Action | 1.5 | ✗
   7 |      89 | Imag |  8.6 | 0.6 |      86 | Imag | Action | 0.8 | ✗
   8 |      88 | Imag |  7.5 | 0.6 |      84 | Imag | Action | 0.7 | ✗
   9 |      68 | Real | 10000.0 | 0.5 |      51 | Real | Action | 0.6 | ✗
  10 |      16 | Real |  4.8 | 0.5 |

## Battery Degradation Physics: How Frequency Relevance Evolves

Interpreting the feature importance changes to understand battery degradation mechanisms across lifecycle stages.

In [20]:
# Analyze frequency relevance evolution across battery lifecycle
print("=== BATTERY DEGRADATION PHYSICS INSIGHTS ===\n")

# Load the comparison data
df_limited = pd.read_csv("../../results/feature_relevance.csv") if os.path.exists("../../results/feature_relevance.csv") else None
df_all_cycles = pd.DataFrame(feature_data)

if df_limited is not None:
    print("🔋 EARLY-STAGE DEGRADATION (Cycles 1-200):")
    early_top_10 = df_limited.head(10)
    early_freqs = []
    for _, row in early_top_10.iterrows():
        if isinstance(row['frequency_hz'], (int, float)):
            early_freqs.append(row['frequency_hz'])
    
    if early_freqs:
        print(f"   • Key frequencies: {[f'{f:.1f}' for f in sorted(early_freqs)[:5]]} Hz")
        print(f"   • Frequency range: {min(early_freqs):.1f} - {max(early_freqs):.1f} Hz")
        print(f"   • Physics: Predictable capacity fade, SEI growth, gradual impedance rise")
    
    print("\n🔋 FULL LIFECYCLE DEGRADATION (Cycles 1-267):")
    full_top_10 = df_all_cycles.head(10)
    full_freqs = []
    for _, row in full_top_10.iterrows():
        if isinstance(row['frequency_hz'], (int, float)):
            full_freqs.append(row['frequency_hz'])
    
    if full_freqs:
        print(f"   • Key frequencies: {[f'{f:.1f}' for f in sorted(full_freqs)[:5]]} Hz")
        print(f"   • Frequency range: {min(full_freqs):.1f} - {max(full_freqs):.1f} Hz")
        print(f"   • Physics: Complex end-of-life effects, lithium plating, thermal runaway onset")
    
    # Analyze specific frequency shifts
    print(f"\n⚡ FREQUENCY EVOLUTION ANALYSIS:")
    
    # Compare low frequency (1-10 Hz) importance
    early_low_freq = len([f for f in early_freqs if 1 <= f <= 10])
    full_low_freq = len([f for f in full_freqs if 1 <= f <= 10])
    
    print(f"   • Low frequency (1-10 Hz) features in top 10:")
    print(f"     - Early stage: {early_low_freq}/10 features")
    print(f"     - Full lifecycle: {full_low_freq}/10 features")
    print(f"     - Change: {'+' if full_low_freq > early_low_freq else ''}{full_low_freq - early_low_freq}")
    
    # Compare action vector importance 
    early_action = len(early_top_10[early_top_10['feature_type'] == 'Action'])
    full_action = len(full_top_10[full_top_10['feature_type'] == 'Action'])
    
    print(f"   • Action vector features in top 10:")
    print(f"     - Early stage: {early_action}/10 features")  
    print(f"     - Full lifecycle: {full_action}/10 features")
    print(f"     - Change: {'+' if full_action > early_action else ''}{full_action - early_action}")
    
    # Physics interpretation
    print(f"\n🧪 DEGRADATION PHYSICS INTERPRETATION:")
    
    if full_low_freq > early_low_freq:
        print(f"   ✓ Low frequencies (1-10 Hz) become MORE important in late-stage")
        print(f"     → Charge transfer resistance dominates end-of-life degradation")
        print(f"     → Active material loss becomes more significant")
    elif full_low_freq < early_low_freq:
        print(f"   ✓ Low frequencies (1-10 Hz) become LESS important in late-stage")
        print(f"     → High-frequency processes (ohmic, SEI) dominate late degradation")
    
    if full_action != early_action:
        action_change = "increases" if full_action > early_action else "decreases" 
        print(f"   ✓ Action vector importance {action_change} in full lifecycle")
        print(f"     → Cycling history becomes {'more' if full_action > early_action else 'less'} critical for late-stage prediction")
    
    # Stability assessment
    matching_features = sum(1 for i in range(min(10, len(df_limited))) 
                          if df_all_cycles.iloc[i]['feature_index'] == df_limited.iloc[i]['feature_index'])
    
    print(f"\n📊 FEATURE STABILITY ASSESSMENT:")
    print(f"   • Only {matching_features}/10 top features remain consistent")
    print(f"   • Late-stage degradation introduces NEW dominant mechanisms")
    print(f"   • Early-stage models may not generalize to end-of-life conditions")
    
    print(f"\n🎯 PRACTICAL IMPLICATIONS:")
    print(f"   1. **Two-Phase Modeling**: Consider separate models for early vs late degradation")
    print(f"   2. **Adaptive Thresholds**: Adjust frequency selection as battery ages")
    print(f"   3. **Physics-Informed**: Use degradation stage to guide feature selection")
    print(f"   4. **Uncertainty Quantification**: Late-stage predictions need larger confidence intervals")

else:
    print("Limited cycles analysis not available for comparison.")
    print("Run the limited cycles analysis first to enable lifecycle comparison.")

=== BATTERY DEGRADATION PHYSICS INSIGHTS ===

🔋 EARLY-STAGE DEGRADATION (Cycles 1-200):

🔋 FULL LIFECYCLE DEGRADATION (Cycles 1-267):
   • Key frequencies: ['0.3', '4.8', '5.6', '7.5', '8.6'] Hz
   • Frequency range: 0.3 - 10000.0 Hz
   • Physics: Complex end-of-life effects, lithium plating, thermal runaway onset

⚡ FREQUENCY EVOLUTION ANALYSIS:
   • Low frequency (1-10 Hz) features in top 10:
     - Early stage: 0/10 features
     - Full lifecycle: 5/10 features
     - Change: +5
   • Action vector features in top 10:
     - Early stage: 2/10 features
     - Full lifecycle: 2/10 features
     - Change: 0

🧪 DEGRADATION PHYSICS INTERPRETATION:
   ✓ Low frequencies (1-10 Hz) become MORE important in late-stage
     → Charge transfer resistance dominates end-of-life degradation
     → Active material loss becomes more significant

📊 FEATURE STABILITY ASSESSMENT:
   • Only 0/10 top features remain consistent
   • Late-stage degradation introduces NEW dominant mechanisms
   • Early-stage 